In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 기본

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier

In [ ]:
# csv 파일 불러오기
file_path = '/content/drive/MyDrive/서울시 빅데이터 공모전/data/processed/df_final_all_preprocessed.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig', low_memory=False)
df.head()

,사용월,호선명,time_group,혼잡,역명_clean,구조평균,구조최소,구조최대,사고수,유입,child_pop_sum,youth_pop_sum,middle_pop_sum,elder_pop_sum,환승량
0,202101,1호선,기타,85553,동대문,12.6625,10.0,19.0,0,109.158033,2.371445,28.573098,38.102411,40.111080,48535.333333
1,202101,1호선,낮시간,227205,동대문,12.6625,10.0,19.0,0,89.818268,4.140349,13.174589,27.672827,44.830502,48535.333333
2,202101,1호선,출근시간,50184,동대문,12.6625,10.0,19.0,0,57.970107,1.030731,8.346038,19.046191,29.547147,48535.333333
3,202101,1호선,퇴근시간,132625,동대문,12.6625,10.0,19.0,0,95.624161,3.856447,21.486049,28.811206,41.470459,48535.333333
4,202101,1호선,기타,34309,동묘앞,6.8375,3.0,12.0,0,70.195318,2.336531,25.357979,22.212263,20.288545,45053.333333


In [ ]:
# 데이터 기본 확인
print(df.shape)
print(df.columns.tolist())
print(df.isna().sum())
df.head()

(73964, 15)
['사용월', '호선명', 'time_group', '혼잡', '역명_clean', '구조평균', '구조최소', '구조최대', '사고수', '유입', 'child_pop_sum', 'youth_pop_sum', 'middle_pop_sum', 'elder_pop_sum', '환승량']
사용월               0
호선명               0
time_group        0
혼잡                0
역명_clean          0
구조평균              0
구조최소              0
구조최대              0
사고수               0
유입                0
child_pop_sum     0
youth_pop_sum     0
middle_pop_sum    0
elder_pop_sum     0
환승량               0
dtype: int64


,사용월,호선명,time_group,혼잡,역명_clean,구조평균,구조최소,구조최대,사고수,유입,child_pop_sum,youth_pop_sum,middle_pop_sum,elder_pop_sum,환승량
0,202101,1호선,기타,85553,동대문,12.6625,10.0,19.0,0,109.158033,2.371445,28.573098,38.102411,40.111080,48535.333333
1,202101,1호선,낮시간,227205,동대문,12.6625,10.0,19.0,0,89.818268,4.140349,13.174589,27.672827,44.830502,48535.333333
2,202101,1호선,출근시간,50184,동대문,12.6625,10.0,19.0,0,57.970107,1.030731,8.346038,19.046191,29.547147,48535.333333
3,202101,1호선,퇴근시간,132625,동대문,12.6625,10.0,19.0,0,95.624161,3.856447,21.486049,28.811206,41.470459,48535.333333
4,202101,1호선,기타,34309,동묘앞,6.8375,3.0,12.0,0,70.195318,2.336531,25.357979,22.212263,20.288545,45053.333333


# 사고수 -> 사고발생 여부로 변환

In [ ]:
# 사고발생 여부 타깃 변수 생성
# 사고수 > 0 이면 실제 사고가 발생한 경우이므로 1
# 사고수 = 0 이면 사고가 발생하지 않은 경우이므로 0
# 즉, 사고 "건수" 예측이 아니라 사고 "발생 여부"를 예측하는 이진분류 문제로 변환

df['사고발생'] = (df['사고수'] > 0).astype(int)

# 사고발생/미발생 데이터 개수 확인
print(df['사고발생'].value_counts())

# 사고발생/미발생 비율 확인
# 클래스 불균형 정도를 확인하기 위한 단계
print(df['사고발생'].value_counts(normalize=True))

사고발생
0    73152
1      812
Name: count, dtype: int64
사고발생
0    0.989022
1    0.010978
Name: proportion, dtype: float64


In [ ]:
# 컬럼명 확인
print(df.columns.tolist())

['사용월', '호선명', 'time_group', '혼잡', '역명_clean', '구조평균', '구조최소', '구조최대', '사고수', '유입', 'child_pop_sum', 'youth_pop_sum', 'middle_pop_sum', 'elder_pop_sum', '환승량', '사고발생']


In [ ]:
# 숫자형 변수 전처리
# 모델 학습에 사용할 주요 변수들을 숫자형으로 변환하는 단계
# CSV를 불러오면 숫자가 문자열로 들어오거나, 쉼표(,)가 포함되어 있을 수 있기 때문에 정리 필요

num_cols = [
    '혼잡', '유입', '구조평균', '구조최소', '구조최대',
    '환승량',
    'child_pop_sum', 'youth_pop_sum', 'middle_pop_sum', 'elder_pop_sum'
]

for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)                         # 문자열로 변환
        .str.replace(',', '', regex=False)   # 숫자 안의 쉼표 제거
        .str.strip()                         # 앞뒤 공백 제거
    )

    # 숫자로 변환
    # 변환이 안 되는 값은 NaN으로 처리
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 변환 과정에서 생긴 결측치 NaN을 각 컬럼의 중앙값으로 대체
# 평균보다 중앙값이 이상치 영향을 덜 받아서 안전함
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# 결측치가 남아 있는지 확인
print(df[num_cols].isna().sum())

혼잡                0
유입                0
구조평균              0
구조최소              0
구조최대              0
환승량               0
child_pop_sum     0
youth_pop_sum     0
middle_pop_sum    0
elder_pop_sum     0
dtype: int64


In [ ]:
# 지수 변수 생성
# 혼잡, 유입, 구조, 환승량은 단위와 크기가 서로 다르기 때문에
# 모델이 비교하기 쉽도록 0~1 범위의 지수로 정규화하는 단계

# 혼잡 값 중 비정상적으로 큰 이상값은 NaN으로 처리
df.loc[df['혼잡'] > 10_000_000, '혼잡'] = np.nan

# 이상값 처리로 생긴 결측치는 중앙값으로 대체
df['혼잡'] = df['혼잡'].fillna(df['혼잡'].median())

# 각 변수를 최댓값으로 나누어 0~1 범위의 지수로 변환
df['혼잡지수'] = df['혼잡'] / df['혼잡'].max()
df['유입지수'] = df['유입'] / df['유입'].max()
df['구조지수'] = df['구조최대'] / df['구조최대'].max()
df['환승지수'] = df['환승량'] / df['환승량'].max()

# 역 주변 유입 인구의 전체 규모 계산
df['전체유입인구'] = (
    df['child_pop_sum'] +
    df['youth_pop_sum'] +
    df['middle_pop_sum'] +
    df['elder_pop_sum']
)

# 연령대별 취약계층 비율 변수 생성
# 고령층과 아동은 사고 위험 대응에서 중요한 정책 대상이 될 수 있음
# 1e-6은 전체유입인구가 0일 때 나누기 오류를 방지하기 위한 아주 작은 값
df['고령층비율'] = df['elder_pop_sum'] / (df['전체유입인구'] + 1e-6)
df['아동비율'] = df['child_pop_sum'] / (df['전체유입인구'] + 1e-6)

# 생성된 지수 변수들의 분포 확인
df[['혼잡지수', '유입지수', '구조지수', '환승지수', '고령층비율', '아동비율']].describe()

,혼잡지수,유입지수,구조지수,환승지수,고령층비율,아동비율
count,73964.000000,73964.000000,73964.000000,73964.000000,73964.000000,73964.000000
mean,0.072063,0.422312,0.411987,0.102789,0.253972,0.072303
std,0.069125,0.172835,0.152661,0.183503,0.089719,0.065628
min,0.000000,0.000000,0.178571,0.000000,0.000000,0.000000
25%,0.029293,0.294796,0.321429,0.000000,0.193796,0.035012
50%,0.051901,0.420421,0.382143,0.000000,0.249059,0.054296
75%,0.089952,0.538274,0.500000,0.154406,0.312207,0.084309
max,1.000000,1.000000,1.000000,1.000000,0.545968,0.479915


In [ ]:
# time_group 원핫인코딩
# time_group은 '출근시간', '퇴근시간', '낮시간' 같은 문자형 범주 변수이기 때문에
# XGBoost 모델에 넣기 위해 0/1 형태의 숫자 변수로 변환

df_model = pd.get_dummies(
    df,
    columns=['time_group'],
    drop_first=False
)

# 원핫인코딩으로 생성된 시간대 컬럼만 따로 추출
time_cols = [col for col in df_model.columns if col.startswith('time_group_')]

# 모델 학습에 사용할 최종 입력 변수 목록
# 혼잡, 유입, 구조, 환승, 취약계층 비율, 시간대 정보를 사용
feature_cols = [
    '혼잡지수',
    '유입지수',
    '구조지수',
    '환승지수',
    '고령층비율',
    '아동비율'
] + time_cols

# X: 모델이 사고발생 여부를 예측할 때 사용할 설명변수
X = df_model[feature_cols]

# y: 모델이 예측해야 하는 정답값
# 사고발생 여부 0/1
y = df_model['사고발생']

# 입력 데이터 크기 확인
print(X.shape)

# 사고발생/미발생 개수 확인
print(y.value_counts())

(73964, 10)
사고발생
0    73152
1      812
Name: count, dtype: int64


# 시간 기준 test/train 분리

In [ ]:
# 2. 시간 기준 train/test split
# train: 2021~2023
# test: 2024~
train_idx = df_model['사용월'] < 202401
test_idx = df_model['사용월'] >= 202401

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

print("train class count")
print(y_train.value_counts())

print("\ntest class count")
print(y_test.value_counts())

train class count
사고발생
0    42335
1      509
Name: count, dtype: int64

test class count
사고발생
0    30817
1      303
Name: count, dtype: int64


# Logistic Regression 학습

In [ ]:
# 입력 변수
X = df_model[feature_cols]

# 타겟 (사고 발생 여부)
y = df_model['사고발생']

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(
    class_weight='balanced',  # 🔥 불균형 처리
    max_iter=1000,
    random_state=42
)

logit.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# 성능 평가

In [ ]:
y_pred = logit.predict(X_test)
y_proba = logit.predict_proba(X_test)[:, 1]

# Risk = 사고 발생 확률
test_result = df_model.loc[test_idx].copy()
test_result['Risk_rf'] = y_proba
test_result['사고발생'] = y_test.values

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred, digits=4))

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

# 시간 기준 test 성능 해석
# Recall = 0.0825
# 2024년 이후 실제 사고 303건 중 25건을 탐지함
# 즉, 이후 시점 사고발생 사례의 약 8.3%를 잡아냄

# ROC-AUC = 0.0442
# 과거 데이터로 학습한 모델이 이후 데이터에서도 사고/비사고를 구분하는 능력 안 좋음

# Precision = 0.0651
# 사고위험으로 예측한 구간 중 실제 사고 비율은 낮음
# 이는 사고발생 데이터가 전체 test 데이터의 약 1% 수준으로 매우 적은 불균형 데이터이기 때문

# Confusion Matrix 해석
# 실제 사고 303건 중 25건을 맞히고 278건을 놓침
# 실제 비사고 30,817건 중 359건을 사고위험으로 예측함

# 결론
# 시간 기반 분할을 적용한 결과, 모델은 Recall 8.3%, ROC-AUC 0.0442를 기록하여
# 과거 데이터를 기반으로 미래 시점의 사고 발생 여부를 효과적으로 구분하지 못하는 것으로 나타났다.

Confusion Matrix
[[30458   359]
 [  278    25]]

Classification Report
              precision    recall  f1-score   support

           0     0.9910    0.9884    0.9897     30817
           1     0.0651    0.0825    0.0728       303

    accuracy                         0.9795     31120
   macro avg     0.5280    0.5354    0.5312     31120
weighted avg     0.9819    0.9795    0.9807     31120

ROC-AUC: 0.8452017022450533
PR-AUC: 0.044157987230553024
Recall: 0.08250825082508251
F1-score: 0.07278020378457059


# Top-K Recall 확인

In [ ]:
# Top-K Recall 계산
# 목적: 모델이 위험하다고 예측한 상위 구간 안에
# 실제 사고발생 사례가 얼마나 포함되는지 확인
# 공모전에서는 전체 구간을 모두 관리하는 것보다
# 위험 상위 구간을 우선 관리하는 전략이 중요하므로 Top-K Recall을 확인함

test_result = X_test.copy()

# 실제 사고발생 여부
test_result['actual'] = y_test.values

# XGBoost가 예측한 사고발생 확률
# 이 값을 Risk Score의 기준으로 사용
test_result['risk_proba'] = y_proba

# 위험도 상위 5%, 10%, 20% 구간별 실제 사고 포함률 계산
for ratio in [0.05, 0.10, 0.20]:
    top_n = int(len(test_result) * ratio)

    # 예측 위험확률이 높은 순서대로 정렬 후 상위 ratio만 선택
    top_risk = (
        test_result
        .sort_values('risk_proba', ascending=False)
        .head(top_n)
    )

    # Top-K Recall = 상위 위험구간 안 실제 사고 수 / 전체 실제 사고 수
    top_recall = top_risk['actual'].sum() / test_result['actual'].sum()

    print(f"Top {int(ratio * 100)}% Recall:", top_recall)
    print(f"Top {int(ratio * 100)}% 안 실제 사고발생 수:", top_risk['actual'].sum())
    print("전체 실제 사고발생 수:", test_result['actual'].sum())
    print("-" * 40)

Top 5% Recall: 0.2607260726072607
Top 5% 안 실제 사고발생 수: 79
전체 실제 사고발생 수: 303
----------------------------------------
Top 10% Recall: 0.5115511551155115
Top 10% 안 실제 사고발생 수: 155
전체 실제 사고발생 수: 303
----------------------------------------
Top 20% Recall: 0.7524752475247525
Top 20% 안 실제 사고발생 수: 228
전체 실제 사고발생 수: 303
----------------------------------------


# 계수 해석

In [ ]:
import pandas as pd
import numpy as np

coef = pd.Series(logit.coef_[0], index=feature_cols)

coef.sort_values(ascending=False)

,0
환승지수,1.006861
구조지수,0.276397
혼잡지수,0.274270
time_group_퇴근시간,0.192227
고령층비율,0.186813
time_group_낮시간,0.114150
아동비율,-0.111089
time_group_출근시간,-0.114588
time_group_기타,-0.191789
유입지수,-0.425270


# Risk Score 붙이기

In [ ]:
# Risk Score 생성
# RandomForest 모델의 사고발생 예측확률을 0~100점 위험도 점수로 변환

# Risk: 사고발생 예측확률, 0~1 사이 값
df_model['Risk'] = logit.predict_proba(X)[:, 1]

# Risk_Score: 해석하기 쉽도록 100점 만점으로 변환
df_model['Risk_Score'] = df_model['Risk'] * 100

# Risk Score 확인
df_model[['역명_clean', '호선명', 'Risk', 'Risk_Score', '사고수', '사고발생']].head()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


,역명_clean,호선명,Risk,Risk_Score,사고수,사고발생
0,동대문,1호선,0.294446,29.444616,0,0
1,동대문,1호선,0.377207,37.720712,0,0
2,동대문,1호선,0.334490,33.448971,0,0
3,동대문,1호선,0.388660,38.865984,0,0
4,동묘앞,1호선,0.286037,28.603732,0,0


In [ ]:
# 원본 df에도 Risk Score 붙이기
# df에는 time_group 컬럼이 남아 있으므로 역·호선·시간대별 위험도 확인 가능

df['Risk'] = df_model['Risk']
df['Risk_Score'] = df_model['Risk_Score']

# 역·호선·시간대별 Risk Score 확인
df[['역명_clean', '호선명', 'time_group', 'Risk', 'Risk_Score', '사고수', '사고발생']].head()

,역명_clean,호선명,time_group,Risk,Risk_Score,사고수,사고발생
0,동대문,1호선,기타,0.294446,29.444616,0,0
1,동대문,1호선,낮시간,0.377207,37.720712,0,0
2,동대문,1호선,출근시간,0.334490,33.448971,0,0
3,동대문,1호선,퇴근시간,0.388660,38.865984,0,0
4,동묘앞,1호선,기타,0.286037,28.603732,0,0


In [ ]:
# 역·호선·시간대별 위험도 순위 생성
# 같은 역/호선/시간대 조합이 여러 행에 존재할 수 있으므로 평균 Risk Score로 집계
# 사고수는 실제 사고 발생 규모 확인을 위해 합계로 집계

top_risk_station_time = (
    df
    .groupby(['역명_clean', '호선명', 'time_group'], as_index=False)
    .agg(
        Risk_Score=('Risk_Score', 'mean'),
        Risk=('Risk', 'mean'),
        사고수=('사고수', 'sum'),
        사고발생=('사고발생', 'max'),
        혼잡지수=('혼잡지수', 'mean'),
        유입지수=('유입지수', 'mean'),
        구조지수=('구조지수', 'mean'),
        환승지수=('환승지수', 'mean')
    )
    .sort_values('Risk_Score', ascending=False)
)

# 위험도 상위 20개 역·호선·시간대 확인
top_risk_station_time.head(20)

,역명_clean,호선명,time_group,Risk_Score,Risk,사고수,사고발생,혼잡지수,유입지수,구조지수,환승지수
331,동대문역사문화공원,4호선,퇴근시간,57.336771,0.573368,0,0,0.103024,0.109965,0.750000,0.849273
703,신도림,2호선,퇴근시간,56.717105,0.567171,17,1,0.279230,0.256430,0.267857,1.000000
701,신도림,2호선,낮시간,56.050980,0.560510,10,1,0.306471,0.203331,0.267857,1.000000
329,동대문역사문화공원,4호선,낮시간,55.458753,0.554588,0,0,0.114660,0.109652,0.750000,0.849273
587,서울,1호선,퇴근시간,53.567390,0.535674,6,1,0.315581,0.126693,0.750000,0.659746
585,서울,1호선,낮시간,52.700034,0.527000,6,1,0.383648,0.115373,0.750000,0.659746
87,고속터미널,7호선,퇴근시간,52.201067,0.522011,8,1,0.085898,0.248755,0.678571,0.699515
327,동대문역사문화공원,2호선,퇴근시간,52.066843,0.520668,0,0,0.087408,0.372661,0.446429,0.849273
702,신도림,2호선,출근시간,51.479498,0.514795,6,1,0.136653,0.078834,0.267857,1.000000
335,동대문역사문화공원,5호선,퇴근시간,51.079036,0.510790,0,0,0.018860,0.502418,0.571429,0.849273


In [ ]:
# 위험 유형 분류
# 각 역·호선·시간대 조합에서 가장 높은 지수를 기준으로 위험 유형을 부여
# 예: 혼잡지수가 가장 높으면 혼잡형, 구조지수가 가장 높으면 구조형

def classify_risk_type(row):
    values = {
        '혼잡형': row['혼잡지수'],
        '유입형': row['유입지수'],
        '구조형': row['구조지수'],
        '환승형': row['환승지수']
    }
    return max(values, key=values.get)

top_risk_station_time['위험유형'] = top_risk_station_time.apply(classify_risk_type, axis=1)

# 위험도 상위 20개 역·호선·시간대와 위험유형 확인
top_risk_station_time.head(20)

,역명_clean,호선명,time_group,Risk_Score,Risk,사고수,사고발생,혼잡지수,유입지수,구조지수,환승지수,위험유형
331,동대문역사문화공원,4호선,퇴근시간,57.336771,0.573368,0,0,0.103024,0.109965,0.750000,0.849273,환승형
703,신도림,2호선,퇴근시간,56.717105,0.567171,17,1,0.279230,0.256430,0.267857,1.000000,환승형
701,신도림,2호선,낮시간,56.050980,0.560510,10,1,0.306471,0.203331,0.267857,1.000000,환승형
329,동대문역사문화공원,4호선,낮시간,55.458753,0.554588,0,0,0.114660,0.109652,0.750000,0.849273,환승형
587,서울,1호선,퇴근시간,53.567390,0.535674,6,1,0.315581,0.126693,0.750000,0.659746,구조형
585,서울,1호선,낮시간,52.700034,0.527000,6,1,0.383648,0.115373,0.750000,0.659746,구조형
87,고속터미널,7호선,퇴근시간,52.201067,0.522011,8,1,0.085898,0.248755,0.678571,0.699515,환승형
327,동대문역사문화공원,2호선,퇴근시간,52.066843,0.520668,0,0,0.087408,0.372661,0.446429,0.849273,환승형
702,신도림,2호선,출근시간,51.479498,0.514795,6,1,0.136653,0.078834,0.267857,1.000000,환승형
335,동대문역사문화공원,5호선,퇴근시간,51.079036,0.510790,0,0,0.018860,0.502418,0.571429,0.849273,환승형
